In [ ]:
#from IPython.core.display import display, HTML
#display(HTML("<style>.container { width:100% !important; }</style>"))


In [ ]:
# Replace the following with your path
%cd "/path/to/your/folder"

In [ ]:
# Load modules
import os
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load .env
load_dotenv()

In [ ]:
# Replace the following with your set up
os.environ['API_ENDPOINT']='XX'
os.environ['API_DEPLOYMENT']='XX'
os.environ['API_VERSION']='XX'
os.environ['API_KEY']='XX' # keep in confidential

client = AzureOpenAI(azure_endpoint=os.getenv('API_ENDPOINT'), api_version=os.getenv('API_VERSION'),
                     api_key=os.getenv('API_KEY'))

In [ ]:
def process_nctid(nctid):
    import os
    from openai import AzureOpenAI
    from dotenv import load_dotenv
    import csv
    import numpy as np
    import json
    import pandas as pd
    from openpyxl import load_workbook, Workbook
    from openpyxl.styles import Font, Alignment
    from openpyxl.utils.dataframe import dataframe_to_rows
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from scipy.spatial.distance import jaccard
    from openpyxl.utils import get_column_letter 
    import sys
    import time

    # Initialize timing and token tracking at function level
    start_time_total = time.time()
    total_input_tokens = 0
    total_output_tokens = 0

    def task1_with_tokens(out_link, prompt1_link):
        nonlocal total_input_tokens, total_output_tokens
        
        # 1.1 Construct prompt
        with open(prompt1_link, 'r') as file:
            prompt1 = file.read()
        
        prompt2 = "# Input" + '\n' + 'The outcome information from the ClinicalTrials.gov registration is:'
        # Information from clinicaltrials.gov
        with open(out_link, 'r') as file:
            data = json.load(file)
        
        prompt3 = str(data)
        
        prompt = prompt1 + "\n" + prompt2 + "\n" + prompt3 + "\n" + prompt1 
        
        # 1.2 Run prompt
        message = [{'role': 'user', 'content': prompt}]
        start_time = time.time()  # Start timing
        
        response = client.chat.completions.create(
            model=os.getenv('API_DEPLOYMENT'), 
            messages=message, 
            seed=6666,
            max_tokens=4096,
            temperature=0,
            top_p=1
        )
        
        end_time = time.time()  # End timing
        elapsed_time = end_time - start_time
        print(f"API processing time for {nctid} (task1): {elapsed_time:.2f} seconds")    
        
        # Extract token usage information
        if hasattr(response, 'usage') and response.usage:
            input_tokens = response.usage.prompt_tokens
            output_tokens = response.usage.completion_tokens
            total_input_tokens += input_tokens
            total_output_tokens += output_tokens
            print(f"Tokens used - Input: {input_tokens}, Output: {output_tokens}")
        else:
            print("Token usage information not available in response")
                
        # 1.3 Save results
        content = response.choices[0].message.content
        table_data = json.loads(content.strip("```json"))
        columns = table_data["table"]["columns"]
        rows = table_data["table"]["rows"]
        output = pd.DataFrame(rows, columns=columns)
        
        return output

    ## Task 1: Parse elements (prospective) ##

    out_link = f"out_pro/{nctid}_merged.json"

    prompt1_link = "Prompts/Define_Compare_Other elements/Task1_metric.txt"
    output2 = task1_with_tokens(out_link, prompt1_link)

    prompt1_link = 'Prompts/Define_Compare_Other elements/Task1_type.txt'
    output3 = task1_with_tokens(out_link, prompt1_link)
    
    output3 = output3.rename(columns={'Actual aggregation method': 'Aggregation method'})

    # Merge the results
    output2['Identifier'] = output2['Identifier'].str.capitalize()
    output3['Identifier'] = output3['Identifier'].str.capitalize()

    task1_res = output2.merge(output3, on='Identifier')

    # Save the definitions to a local file
    # Load the JSON data
    out_link = f"out_pro/{nctid}_merged.json"
    with open(out_link, 'r') as file:
        data = json.load(file)

    # Create a DataFrame
    df = pd.DataFrame(data)
    
    # Add outcome measure, description, timeframe etc.
    result1 = task1_res.loc[:, ['Identifier',"Rationale, Metric","Metric","Rationale, Actual aggregation method","Aggregation method"]]

    result1 = pd.merge(result1, df, on='Identifier', how='outer')

    # Add nctid, link
    inf_link = f"inf_pro/{nctid}.json"

    with open(inf_link, 'r') as file:
        data = json.load(file)
    v1 = data['Version Before Recruiting']+1
    v2 = data['Most Recent Version']+1

    result1['index'] = result1.index + 1
    result1['record_id'] = nctid
    result1['out_link'] = f'https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#outcome-measures-card'
    result1['res_link'] = f"https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#results-outcome-measures-card"

    # Workbook file path
    workbook_path = "metric_aggregation_method_definition_prospective.xlsx"

    # Check if the workbook file exists
    if os.path.exists(workbook_path):
        # Load the existing workbook
        workbook = load_workbook(filename=workbook_path)
        if 'Comparisons' not in workbook.sheetnames:
            sheet = workbook.create_sheet('Comparisons')
        else:
            sheet = workbook['Comparisons']
    else:
        # Create a new workbook
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = 'Comparisons'

    # Get the next empty row
    next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1

    # Write the header row if it's the first entry
    if next_row == 1:
        for col_num, column_title in enumerate(result1.columns, 1):
            cell = sheet.cell(row=1, column=col_num)
            cell.value = column_title
            cell.font = Font(bold=True)

    # Write the data rows
    for row in dataframe_to_rows(result1, index=False, header=False):
        sheet.append(row)

    # Adjust cell size and text wrapping
    for row in sheet.iter_rows(min_row=next_row, max_row=sheet.max_row):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True)

    # Set column widths to a fixed value
    fixed_width = 20  # Specify the desired fixed width
    for col in sheet.columns:
        sheet.column_dimensions[col[0].column_letter].width = fixed_width

    # Save the workbook
    workbook.save(workbook_path)

    ## Task 2: Parse elements (last) ##

    out_link = f"out_last_metric/{nctid}_merged.json"

    prompt1_link = 'Prompts/Define_Compare_Other elements/Task2_metric.txt'
    output2 = task1_with_tokens(out_link, prompt1_link)

    out_link = f"out_last_agg/{nctid}_merged.json"

    prompt1_link = 'Prompts/Define_Compare_Other elements/Task2_type.txt'
    output3 = task1_with_tokens(out_link, prompt1_link)
    
    output3 = output3.rename(columns={'Actual aggregation method': 'Aggregation method'})

    # Merge the results
    output2['Identifier'] = output2['Identifier'].str.capitalize()
    output3['Identifier'] = output3['Identifier'].str.capitalize()

    task2_res = output2.merge(output3, on='Identifier')

    # Save the definitions to a local file
    # Load the JSON data
    out_link = f"out_last_agg/{nctid}_merged.json"
    with open(out_link, 'r') as file:
        data = json.load(file)

    # Create a DataFrame
    df = pd.DataFrame(data)

    # Save the parsed definition
    # Add outcome measure, description, timeframe etc.
    result2 = task2_res.loc[:, ['Identifier',"Rationale, Metric","Metric","Rationale, Actual aggregation method","Aggregation method"]]

    result2 = pd.merge(result2, df, on='Identifier', how='outer')

    # Add nctid, link
    inf_link = f"inf_pro/{nctid}.json"

    with open(inf_link, 'r') as file:
        data = json.load(file)
    v1 = data['Version Before Recruiting']+1
    v2 = data['Most Recent Version']+1

    result2['index'] = result2.index + 1
    result2['record_id'] = nctid
    result2['out_link'] = f'https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#outcome-measures-card'
    result2['res_link'] = f"https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#results-outcome-measures-card"

    # Workbook file path
    workbook_path = "metric_aggregation_method_definition_latest.xlsx"

    # Check if the workbook file exists
    if os.path.exists(workbook_path):
        # Load the existing workbook
        workbook = load_workbook(filename=workbook_path)
        if 'Comparisons' not in workbook.sheetnames:
            sheet = workbook.create_sheet('Comparisons')
        else:
            sheet = workbook['Comparisons']
    else:
        # Create a new workbook
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = 'Comparisons'

    # Get the next empty row
    next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1

    # Write the header row if it's the first entry
    if next_row == 1:
        for col_num, column_title in enumerate(result2.columns, 1):
            cell = sheet.cell(row=1, column=col_num)
            cell.value = column_title
            cell.font = Font(bold=True)

    # Write the data rows
    for row in dataframe_to_rows(result2, index=False, header=False):
        sheet.append(row)

    # Adjust cell size and text wrapping
    for row in sheet.iter_rows(min_row=next_row, max_row=sheet.max_row):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True)

    # Set column widths to a fixed value
    fixed_width = 20  # Specify the desired fixed width
    for col in sheet.columns:
        sheet.column_dimensions[col[0].column_letter].width = fixed_width

    # Save the workbook
    workbook.save(workbook_path)
    
    ## Task 3: Import matching results ##
    
    output3 = pd.read_csv(f"{nctid}_table3.csv")

    # GPT might mistakenly enter [Missing outcome] for [Added outcome], and vice versa
    output3['Outcome (prospective)'] = output3['Outcome (prospective)'].replace('[Missing outcome]', '[Added outcome]')
    output3['Outcome (last)'] = output3['Outcome (last)'].replace('[Added outcome]', '[Missing outcome]')

    output3['Identifier (prospective)'] = output3['Identifier (prospective)'].replace('[Missing outcome]', '[Added outcome]')
    output3['Identifier (last)'] = output3['Identifier (last)'].replace('[Added outcome]', '[Missing outcome]')

    # Combine output tables from tasks 1, 2, and 3

    # Add prospective/last to each variable name
    table1 = task1_res.rename(columns=lambda x: f"{x} (prospective)")
    table2 = task2_res.rename(columns=lambda x: f"{x} (last)")

    # Merge the tables from 1 and 2 and the matched table from 3
    combined_table = pd.merge(table1, output3, on='Identifier (prospective)', how='outer')
    combined_table = pd.merge(combined_table, table2, on='Identifier (last)', how='outer')
    
    # Keep only the matched outcomes

    # Create a boolean mask to identify the rows to keep
    mask = ~ ((combined_table['Outcome (last)'] == '[Missing outcome]') | pd.isna(combined_table['Outcome (last)']) |
    (combined_table['Outcome (prospective)'] == '[Added outcome]') | pd.isna(combined_table['Outcome (prospective)']))

    # Filter the DataFrame to keep only the desired rows
    matched = combined_table[mask]

    # Move the unmatched outcomes to a separate dataframe
    unmatched_df = combined_table[~mask]

    # If all outcomes are unmatched, save the results and exit
    if matched.empty:  
        print("No matched outcomes")
        result = unmatched_df.loc[:, ['Identifier (prospective)','Outcome (prospective)','Identifier (last)','Outcome (last)']]
        
        # Calculate final timing and save processing info
        end_time_total = time.time()
        elapsed_time_total = end_time_total - start_time_total
        
        # Save processing time and token usage to Excel
        time_workbook_path = "processing_cost_metric_aggregation_method_timepoint.xlsx"

        # Check if the time workbook file exists
        if os.path.exists(time_workbook_path):
            time_workbook = load_workbook(filename=time_workbook_path)
            if 'ProcessingTime' not in time_workbook.sheetnames:
                time_sheet = time_workbook.create_sheet('ProcessingTime')
                time_sheet.append(['nctid', 'processing_time_seconds', 'input_tokens', 'output_tokens', 'total_tokens'])
            else:
                time_sheet = time_workbook['ProcessingTime']
                # Check if we need to add new columns to existing sheet
                if time_sheet.max_column < 5:
                    header_row = [cell.value for cell in time_sheet[1]]
                    if 'input_tokens' not in header_row:
                        time_sheet.cell(row=1, column=3, value='input_tokens')
                    if 'output_tokens' not in header_row:
                        time_sheet.cell(row=1, column=4, value='output_tokens')
                    if 'total_tokens' not in header_row:
                        time_sheet.cell(row=1, column=5, value='total_tokens')
        else:
            # Create new workbook
            time_workbook = Workbook()
            time_sheet = time_workbook.active
            time_sheet.title = 'ProcessingTime'
            time_sheet.append(['nctid', 'processing_time_seconds', 'input_tokens', 'output_tokens', 'total_tokens'])

        # Append this nctid and its elapsed time and token usage
        total_tokens = total_input_tokens + total_output_tokens
        time_sheet.append([nctid, elapsed_time_total, total_input_tokens, total_output_tokens, total_tokens])

        # Save time workbook
        time_workbook.save(time_workbook_path)
        
        return result
           
    ### For matched outcomes ###

    ## Task 4: Compare timepoints ##
    
    # Add original texts
    
    out_link = f"out_pro/{nctid}_merged.json"

    with open(out_link, 'r') as file:
        data = json.load(file)

    df1 = pd.DataFrame({'Original text (prospective)': data})

    # Create identifier
    def extract_priority_and_number(data):
        identifier = data.get('Identifier', '')
        return identifier.strip()

    # Apply the function to create new column
    df1['Identifier (prospective)'] = df1['Original text (prospective)'].apply(extract_priority_and_number).str.capitalize()

    out_link = f"out_last_time/{nctid}_merged.json"

    with open(out_link, 'r') as file:
        data = json.load(file)

    df2 = pd.DataFrame({'Original text (last)': data})

    # Add the Identifier column
    df2['Identifier (last)'] = df2['Original text (last)'].apply(extract_priority_and_number).str.capitalize()

    combined_table_tx = pd.merge(matched, df1, on='Identifier (prospective)', how='inner')
    combined_table_tx = pd.merge(combined_table_tx, df2, on='Identifier (last)', how='inner')
    
    # Specify the columns you want to keep
    columns_to_keep = ['Identifier (prospective)', 'Original text (prospective)', 'Identifier (last)', 'Original text (last)']

    # Create a new DataFrame with only the specified columns
    matched_tx = combined_table_tx.loc[:, columns_to_keep]

    matched_tx_string = matched_tx.to_json(orient='records')

    # 5.1  Construct prompt
    with open("Prompts/Define_Compare_Other elements/Task4_time.txt", 'r') as file:
        # Read the contents of the file
        prompt1 = file.read()

    prompt2 = "Input" + "\n" + "Here is the combined table containing original text of outcomes in both the prospective and last versions from ClinicalTrials.gov:"
    
    prompt3 = matched_tx_string

    prompt = prompt1 + "\n" + prompt2 + "\n" + prompt3 + "\n" + prompt1

    # 5.2 Run prompt
    message = [{'role': 'user', 'content': prompt}]
    start_time = time.time()  # Start timing

    response4 = client.chat.completions.create(
        model=os.getenv('API_DEPLOYMENT'), 
        messages=message, 
        seed=66,
        max_tokens=4096,
        temperature=0,
        top_p=1
    )
    
    end_time = time.time()  # End timing
    elapsed_time = end_time - start_time
    print(f"API processing time for {nctid} (task4): {elapsed_time:.2f} seconds")

    # Extract token usage information for Task 4
    if hasattr(response4, 'usage') and response4.usage:
        input_tokens = response4.usage.prompt_tokens
        output_tokens = response4.usage.completion_tokens
        total_input_tokens += input_tokens
        total_output_tokens += output_tokens
        print(f"Task 4 Tokens used - Input: {input_tokens}, Output: {output_tokens}")
    else:
        print("Task 4 Token usage information not available in response")

    # 5.3 Save results to a local file

    # Extract the content part from the OpenAIObject
    content4 = response4.choices[0].message.content

    # Parse the content string as JSON
    table_data4 = json.loads(content4.strip("```json"))

    # Extract the table columns and rows
    columns = table_data4["table"]["columns"]
    rows = table_data4["table"]["rows"]

    # Create a DataFrame from the rows and columns
    table_t = pd.DataFrame(rows, columns=columns)

    # Combine with the previous results
    result = pd.merge(table_t, combined_table, on='Identifier (prospective)', how='outer')

    ## Task 5: Compare remaining elements ##
    
    # 5.1 Compare Metric
    # Initialize the 'Changes (Metric)' column with default values
    result['Changes (Metric)'] = 'No change'

    # Update 'Changes (Metric)' to 'Change in definition' where 'Metric' is 'Different'
    result.loc[(result['Metric (prospective)']!=result['Metric (last)']), 'Changes (Metric)'] = 'Change in definition'

    # Update to 'More complete' based on specific conditions
    result.loc[((result['Metric (prospective)'] == 'Not reported') | 
                (result['Metric (prospective)'] == "Not valid")) &
               (result['Metric (last)'] != "Not reported") & (result['Metric (last)'] != "Not valid"),
               'Changes (Metric)'] = 'More complete'

    # Update to 'Less complete' based on specific conditions
    result.loc[((result['Metric (last)'] == 'Not reported') | (result['Metric (last)'] == "Not valid")) & 
               (result['Metric (prospective)'] != "Not reported") & 
               (result['Metric (prospective)'] != "Not valid"), 
        'Changes (Metric)'] = 'Less complete'
    
    # 5.2 Compare Aggregation method
    # Initialize 'Changes (Aggregation method)' with 'Change in definition'
    result['Changes (Aggregation method)'] = 'Change in definition'

    # Update to 'More complete' where 'Aggregation method (prospective)' is 'Not reported'
    result.loc[
        (result['Aggregation method (prospective)'] != result['Aggregation method (last)']) & 
        (result['Aggregation method (prospective)'] == "Not reported"), 
        'Changes (Aggregation method)'
    ] = 'More complete'

    # Update to 'Less complete' where 'Aggregation method (last)' is 'Not reported'
    result.loc[
        (result['Aggregation method (prospective)'] != result['Aggregation method (last)']) & 
        (result['Aggregation method (last)'] == "Not reported"), 
        'Changes (Aggregation method)'
    ] = 'Less complete'

    # Update to 'No change' where 'Aggregation method (prospective)' equals 'Aggregation method (last)'
    result.loc[
        result['Aggregation method (prospective)'] == result['Aggregation method (last)'], 
        'Changes (Aggregation method)'
    ] = 'No change'

    # Change to no values for the unmatched outcomes
    result.loc[result['Changes (Timepoint)'].isna(), 'Changes (Metric)'] = np.nan
    result.loc[result['Changes (Timepoint)'].isna(), 'Changes (Aggregation method)'] = np.nan

    # Desired column order
    column_order = [
        "Identifier (prospective)", "Outcome (prospective)", "Identifier (last)", "Outcome (last)",
        "Rationale, Metric (prospective)", "Metric (prospective)", "Rationale, Metric (last)", "Metric (last)", "Changes (Metric)",
        "Rationale, Actual aggregation method (prospective)", "Aggregation method (prospective)", 
        "Rationale, Acutual aggregation method (last)", "Aggregation method (last)", "Changes (Aggregation method)", 
        "Rationale, Timepoint", "Changes (Timepoint)"
    ]

    # Reorder the DataFrame columns
    result = result.reindex(columns=column_order)

    # Save all results
    result.to_csv(f"{nctid}_comb.csv", index=False, encoding='utf-8-sig')
    
    ## Save the comparison results ## 
    result3 = result.loc[:, ['Identifier (prospective)','Outcome (prospective)','Identifier (last)','Outcome (last)',
                             'Changes (Metric)','Changes (Aggregation method)','Changes (Timepoint)']]

    # Add nctid, link
    inf_link = f"inf_pro/{nctid}.json"

    with open(inf_link, 'r') as file:
        data = json.load(file)
    v1 = data['Version Before Recruiting']+1
    v2 = data['Most Recent Version']+1

    result3['index'] = result3.index + 1
    result3['record_id'] = nctid
    result3['out_link'] = f'https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#outcome-measures-card'
    result3['res_link'] = f"https://clinicaltrials.gov/study/{nctid}?tab=history&a={v1}&b={v2}&compareMode=sideBySide#results-outcome-measures-card"

    # Save results to a local file

    # Workbook file path
    workbook_path = "metric_aggregation_method_timepoint_change_status.xlsx"

    # Check if the workbook file exists
    if os.path.exists(workbook_path):
        # Load the existing workbook
        workbook = load_workbook(filename=workbook_path)
        if 'Comparisons' not in workbook.sheetnames:
            sheet = workbook.create_sheet('Comparisons')
        else:
            sheet = workbook['Comparisons']
    else:
        # Create a new workbook
        workbook = Workbook()
        sheet = workbook.active
        sheet.title = 'Comparisons'

    # Get the next empty row
    next_row = sheet.max_row + 1 if sheet.max_row > 1 else 1

    # Write the header row if it's the first entry
    if next_row == 1:
        for col_num, column_title in enumerate(result3.columns, 1):
            cell = sheet.cell(row=1, column=col_num)
            cell.value = column_title
            cell.font = Font(bold=True)

    # Write the data rows
    for row in dataframe_to_rows(result3, index=False, header=False):
        sheet.append(row)

    # Adjust cell size and text wrapping
    for row in sheet.iter_rows(min_row=next_row, max_row=sheet.max_row):
        for cell in row:
            cell.alignment = Alignment(wrap_text=True)

    # Set column widths to a fixed value
    fixed_width = 20  # Specify the desired fixed width
    for col in sheet.columns:
        sheet.column_dimensions[col[0].column_letter].width = fixed_width

    # Save the workbook
    workbook.save(workbook_path)

    # Calculate final timing
    end_time_total = time.time()
    elapsed_time_total = end_time_total - start_time_total

    # Save processing time and token usage to Excel
    time_workbook_path = "processing_cost_metric_aggregation_method_timepoint.xlsx"

    # Check if the time workbook file exists
    if os.path.exists(time_workbook_path):
        time_workbook = load_workbook(filename=time_workbook_path)
        if 'ProcessingTime' not in time_workbook.sheetnames:
            time_sheet = time_workbook.create_sheet('ProcessingTime')
            time_sheet.append(['nctid', 'processing_time_seconds', 'input_tokens', 'output_tokens', 'total_tokens'])
        else:
            time_sheet = time_workbook['ProcessingTime']
            # Check if we need to add new columns to existing sheet
            if time_sheet.max_column < 5:
                header_row = [cell.value for cell in time_sheet[1]]
                if 'input_tokens' not in header_row:
                    time_sheet.cell(row=1, column=3, value='input_tokens')
                if 'output_tokens' not in header_row:
                    time_sheet.cell(row=1, column=4, value='output_tokens')
                if 'total_tokens' not in header_row:
                    time_sheet.cell(row=1, column=5, value='total_tokens')
    else:
        # Create new workbook
        time_workbook = Workbook()
        time_sheet = time_workbook.active
        time_sheet.title = 'ProcessingTime'
        time_sheet.append(['nctid', 'processing_time_seconds', 'input_tokens', 'output_tokens', 'total_tokens'])

    # Append this nctid and its elapsed time and token usage
    total_tokens = total_input_tokens + total_output_tokens
    time_sheet.append([nctid, elapsed_time_total, total_input_tokens, total_output_tokens, total_tokens])

    # Save time workbook
    time_workbook.save(time_workbook_path)

    print(f"Total processing time for {nctid}: {elapsed_time_total:.2f} seconds")
    print(f"Total tokens used - Input: {total_input_tokens}, Output: {total_output_tokens}, Total: {total_tokens}")

    return result

In [ ]:
import time

# Read NCTIDs from a text file
with open('nctid_new150.txt', 'r') as file:
    nctids = [line.strip() for line in file if line.strip()]

# Process each NCTID
# Initialize a counter variable
n = 1

for nctid in nctids:
    print(f"Processing NCTID {n}/{len(nctids)}: {nctid}")
    process_nctid(nctid)
    print(f"Finished processing NCTID: {nctid}")
    
    # Optional: Add a delay between processing each NCTID to avoid rate limiting
    time.sleep(5)
    
    # Increment the counter
    n += 1

print("All NCTIDs processed.")

In [ ]:
import sys

# Stop the script
sys.exit()